# Ground Truth 验证系统 v2

从本地 Excel 数据计算指标，与后端 API 结果对比，验证后端计算逻辑正确性。

**重要更新**：
- 移除 building_id/system_id 筛选（数据库中无此字段）
- 修复参数文件过滤问题
- 强制重新加载所有模块

In [ ]:
# 强制重新加载模块
import sys
from pathlib import Path

# 清除已加载的模块
modules_to_reload = [k for k in sys.modules.keys() if 'ground_truth_validation' in k]
for mod in modules_to_reload:
    del sys.modules[mod]

print(f"已清除 {len(modules_to_reload)} 个缓存模块")

In [ ]:
from datetime import datetime
sys.path.insert(0, str(Path.cwd().parent))

from ground_truth_validation.config import (
    DATA_DIR, BUILDING_ID, SYSTEM_ID, CALIBRATION_MONTH,
    EPSILON_TOLERANCE, BACKEND_URL, ALL_METRICS
)
from ground_truth_validation.excel_reader import ExcelReader
from ground_truth_validation.backend_client import BackendAPIClient
from ground_truth_validation.orchestrator import MetricOrchestrator
from ground_truth_validation.comparison import ComparisonEngine
from ground_truth_validation.report import ReportGenerator

print("模块导入完成")
print(f"BUILDING_ID: {BUILDING_ID}")
print(f"SYSTEM_ID: {SYSTEM_ID}")

In [ ]:
reader = ExcelReader(DATA_DIR)
backend = BackendAPIClient(BACKEND_URL)
orchestrator = MetricOrchestrator(reader)
comparator = ComparisonEngine(EPSILON_TOLERANCE)
reporter = ReportGenerator()

print("组件初始化完成")

In [ ]:
if backend.health_check():
    print("✓ 后端 API 可访问")
else:
    print("✗ 后端 API 无法访问")
    print("请先启动后端: uvicorn carbon_metrics.backend.main:app --reload")

In [ ]:
time_start = datetime.fromisoformat(f"{CALIBRATION_MONTH}-01T00:00:00")
time_end = datetime.fromisoformat(f"{CALIBRATION_MONTH}-31T23:59:59")

print(f"时间范围: {time_start} 至 {time_end}")
print(f"建筑: {BUILDING_ID}, 系统: {SYSTEM_ID}")
print(f"待验证指标: {len(ALL_METRICS)} 个")

In [ ]:
# 仅验证第一个指标（系统总电量）
metric_name = "系统总电量"
print(f"\n验证指标: {metric_name}")

try:
    # GT 计算
    print("  [1/2] 计算 Ground Truth...")
    gt_result = orchestrator.calculate_metric(metric_name, time_start, time_end, BUILDING_ID, SYSTEM_ID)
    print(f"  GT 值: {gt_result.get('value')}")
    print(f"  GT 状态: {gt_result.get('status')}")
    
    # 后端计算
    print("  [2/2] 调用后端 API...")
    backend_result = backend.calculate_metric(
        metric_name, time_start, time_end,
        building_id=BUILDING_ID,
        system_id=SYSTEM_ID
    )
    print(f"  后端值: {backend_result.get('value')}")
    print(f"  后端状态: {backend_result.get('status')}")
    
    # 对比
    comparison = comparator.compare(gt_result, backend_result)
    print(f"\n  对比结果: {'✓ 匹配' if comparison['match'] else '✗ 不匹配'}")
    if not comparison['match']:
        print(f"  差异: {comparison.get('diff')}")
        print(f"  差异百分比: {comparison.get('diff_pct')}%")
        
except Exception as e:
    print(f"  ✗ 错误: {e}")
    import traceback
    traceback.print_exc()

## 诊断信息

如果后端返回 `no_data`，检查后端诊断信息：

In [ ]:
if 'backend_result' in locals() and backend_result.get('status') == 'no_data':
    print("后端诊断信息:")
    print(f"  总记录数: {backend_result['trace']['data_source']['total_records']}")
    print(f"  有效记录数: {backend_result['trace']['data_source']['valid_records']}")
    
    if backend_result.get('quality_issues'):
        for issue in backend_result['quality_issues']:
            if issue['type'] == 'missing_dependency':
                details = issue.get('details', {})
                diag = details.get('missing_metric_diagnostics', {})
                for metric, info in diag.items():
                    print(f"\n  指标 '{metric}' 诊断:")
                    print(f"    筛选范围内: {info.get('agg_scope_count', 0)} 条")
                    print(f"    全局: {info.get('agg_global_count', 0)} 条")
                    print(f"    原因: {info.get('reason', 'unknown')}")